In [ ]:
import os
import random
import shutil
from pathlib import Path

sets = [25, 50, 100]
for s in sets:
    INPUT_DIR = "input_data/new_datasets/100/train"
    OUTPUT_DIR = "input_data/new_datasets/nowe/50"
    PERCENT_USE = 50

    input_path = Path(INPUT_DIR)
    output_path = Path(OUTPUT_DIR)

    for class_dir in input_path.iterdir():
        if not class_dir.is_dir:
            continue

        class_name = class_dir.name
        print(class_name, "being proccesed")
        images = list(class_dir.glob("*"))

        num_to_use = int(len(images) * PERCENT_USE)
        images = images[:num_to_use]

        split = {"train": images}

        for split_name, image_list in split.items():
            target_dir = output_path / split_name / class_name
            target_dir.mkdir(parents=True, exist_ok=True)

            for image_path in image_list:
                shutil.copy(image_path, target_dir)

In [ ]:
import os
import random
import shutil
from pathlib import Path
import cv2
import numpy as np
import random
from PIL import Image

sets = [25, 75, 50]

for s in sets:
    INPUT_DIR = "nowe/100/train"
    OUTPUT_DIR = f"nowe/noise/{s}/train"
    PERCENT_NOISE = s / 100

    input_path = Path(INPUT_DIR)
    output_path = Path(OUTPUT_DIR)

    for class_dir in input_path.iterdir():
        if not class_dir.is_dir:
            continue

        class_name = class_dir.name
        print(class_name, "being proccesed")
        images = list(class_dir.glob("*"))

        train_images = images

        num_to_blur = int(len(train_images) * PERCENT_NOISE)
        blur_id = set(random.sample(range(len(train_images)), num_to_blur))

        target_dir = output_path / class_name
        target_dir.mkdir(parents=True, exist_ok=True)

        for idx, image_path in enumerate(train_images):
            target_path = target_dir / image_path.name
            if idx in blur_id:
                img = Image.open(image_path)
                motion_strength = random.randint(5, 30)
                direction = random.choice(["horizontal", "vertical"])
                kernel = np.zeros((motion_strength, motion_strength))
                if direction == "horizontal":
                    kernel[int((motion_strength - 1) // 2), :] = np.ones(
                        motion_strength
                    )
                else:
                    kernel[:, int((motion_strength - 1) // 2)] = np.ones(
                        motion_strength
                    )

                kernel = kernel / motion_strength

                img_np = np.array(img)

                motion_blur = cv2.filter2D(img_np, -1, kernel)
                noise_strength = random.randint(5, 20)

                noise = np.random.normal(0, noise_strength, motion_blur.shape)

                noisy_img = motion_blur + noise

                noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)

                final_img = Image.fromarray(noisy_img)

                final_img.save(target_path)

            else:
                shutil.copy(image_path, target_dir)